# FASTA2GO - Etapa 1: validação da extração

Confere, **somente leitura**, o banco produzido por `etapa1/main.py`: alinhamento das tabelas,
denominadores das features, anotações GO e resumo final.

```bash
uv run python etapa1/main.py     # constrói db/fasta2go.duckdb (ajustes em etapa1/fasta2go.config)
```

Nada é gravado: a base é aberta em modo somente leitura e a execução para na primeira
inconsistência (`assert`).


In [1]:
import sys
from pathlib import Path

# A raiz do projeto é localizada por marcador: o kernel abre com cwd na pasta do notebook.
ROOT = next(p for p in Path.cwd().resolve().parents if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from fasta2go import db, features, pipeline  # noqa: E402

DATABASE = ROOT / "db" / "fasta2go.duckdb"
assert DATABASE.exists(), f"base ausente: rode `uv run python etapa1/main.py` ({DATABASE})"

conn = db.connect(DATABASE, read_only=True)


def show(sql: str) -> None:
    """Imprime o resultado de uma consulta como tabela simples."""
    result = conn.execute(sql)
    print(" | ".join(column[0] for column in result.description))
    for row in result.fetchall():
        print(" | ".join("" if value is None else str(value) for value in row))


print(f"banco: {DATABASE}")


banco: /home/orfeu/Documents/github/FASTA2GO/db/fasta2go.duckdb


## Contagens e alinhamento

As três tabelas de features, a de anotações e a de GO precisam ter uma linha por proteína, e
nenhuma linha pode apontar para uma proteína que não existe.


In [2]:
counts = conn.execute("""
    SELECT
        (SELECT count(*) FROM proteins) AS proteins,
        (SELECT count(*) FROM sequence_features) AS features,
        (SELECT count(*) FROM amino_acid_composition) AS aac,
        (SELECT count(*) FROM dipeptide_composition) AS dpc,
        (SELECT count(*) FROM uniprot_annotations) AS annotate,
        (SELECT count(*) FROM protein_go_annotations) AS go_rows,
        (SELECT count(*) FROM ingestion_issues) AS issues
""").fetchone()
print(counts)

proteins, feature_rows, aac_rows, dpc_rows, annotated, _, _ = counts
assert proteins > 0
assert proteins == feature_rows == aac_rows == dpc_rows
assert annotated <= proteins
print(f"uma linha por proteína nas quatro tabelas de features: {proteins:,}")

orphans = conn.execute("""
    SELECT
        (SELECT count(*) FROM sequence_features f
           LEFT JOIN proteins p USING (source_file, fasta_id) WHERE p.fasta_id IS NULL)
      + (SELECT count(*) FROM amino_acid_composition a
           LEFT JOIN proteins p USING (source_file, fasta_id) WHERE p.fasta_id IS NULL)
      + (SELECT count(*) FROM dipeptide_composition d
           LEFT JOIN proteins p USING (source_file, fasta_id) WHERE p.fasta_id IS NULL)
      + (SELECT count(*) FROM protein_go_annotations g
           LEFT JOIN proteins p USING (source_file, fasta_id) WHERE p.fasta_id IS NULL)
""").fetchone()[0]
assert orphans == 0
print(f"linhas órfãs: {orphans}")


(2000, 2000, 2000, 2000, 2000, 6872, 15)
uma linha por proteína nas quatro tabelas de features: 2,000
linhas órfãs: 0


## Denominadores

Cada percentual tem denominador explícito: AAC soma 100 sobre os resíduos canônicos e DPC soma
100 sobre os pares adjacentes canônicos. O DPC é indefinido (NULL) exatamente quando a sequência
não tem nenhum par canônico.


In [3]:
for table, columns in (
    ("amino_acid_composition", features.AAC_COLUMNS),
    ("dipeptide_composition", features.DPC_COLUMNS),
):
    expression = " + ".join(f'"{column}"' for column in columns)
    outside = conn.execute(
        f"SELECT count(*) FROM {table} WHERE abs(({expression}) - 100) > 1e-6"
    ).fetchone()[0]
    assert outside == 0, table

print(f"AAC: {len(features.AAC_COLUMNS)} colunas, todas as linhas somam 100")
print(f"DPC: {len(features.DPC_COLUMNS)} colunas, todas as linhas definidas somam 100")

undefined_without_pairs = conn.execute("""
    SELECT count(*) FROM sequence_features s
    JOIN dipeptide_composition d USING (source_file, fasta_id)
    WHERE (s.canonical_pairs = 0) <> (d.A_A IS NULL)
""").fetchone()[0]
assert undefined_without_pairs == 0
print("DPC definido exatamente quando existe par canônico")

bad_length = conn.execute("""
    SELECT count(*) FROM proteins p JOIN sequence_features s USING (source_file, fasta_id)
    WHERE s.canonical_length + s.noncanonical_residues <> p.sequence_length
""").fetchone()[0]
assert bad_length == 0
print("canonical_length + noncanonical_residues = sequence_length em todas as linhas")


AAC: 20 colunas, todas as linhas somam 100
DPC: 400 colunas, todas as linhas definidas somam 100
DPC definido exatamente quando existe par canônico
canonical_length + noncanonical_residues = sequence_length em todas as linhas


## Features da sequência

Amostra das propriedades calculadas e a versão das fórmulas que as produziu.


In [4]:
show("""
    SELECT fasta_id, sequence_length, canonical_length, noncanonical_residues,
           round(molecular_weight, 1) AS mw, round(isoelectric_point, 2) AS pi,
           round(gravy, 3) AS gravy, round(sequence_entropy, 3) AS entropia
    FROM protein_features ORDER BY fasta_id LIMIT 5
""")
print()
show("""
    SELECT feature_version, count(*) AS proteinas,
           min(canonical_length) AS menor, round(avg(canonical_length), 1) AS media,
           max(canonical_length) AS maior
    FROM sequence_features GROUP BY 1
""")


fasta_id | sequence_length | canonical_length | noncanonical_residues | mw | pi | gravy | entropia
sp|A0A097SRX3|3004L_ASFB7 | 328 | 328 | 0 | 39514.1 | 9.26 | -0.248 | 4.153
sp|A0A0N9HQ36|2ODD_SINHE | 310 | 310 | 0 | 35027.6 | 5.36 | -0.321 | 4.176
sp|A0A0P0UZP7|1A15_ORYSJ | 533 | 533 | 0 | 58163.7 | 6.92 | -0.034 | 4.125
sp|A0A0P0WIY3|1A13_ORYSJ | 452 | 452 | 0 | 49321.4 | 6.1 | -0.107 | 4.03
sp|A0A0U5AUY6|3SA9_NAJNA | 72 | 72 | 0 | 8040.8 | 9.15 | 0.343 | 3.717

feature_version | proteinas | menor | media | maior
2 | 2000 | 10 | 231.9 | 2432


## Anotações e GO

Contagem por aspecto, termos de função molecular mais frequentes e as duas garantias de
associação: a anotação é do accession da própria proteína e um accession que a fonte não
devolveu não recebe termo algum.


In [5]:
show("""
    SELECT aspect, count(*) AS anotacoes, count(DISTINCT go_id) AS termos
    FROM protein_go_annotations GROUP BY 1 ORDER BY 2 DESC
""")
print()
show("""
    SELECT go_id, any_value(go_name) AS termo, count(*) AS proteinas
    FROM protein_go_annotations WHERE aspect = 'molecular_function'
    GROUP BY 1 ORDER BY 3 DESC LIMIT 5
""")

leaked = conn.execute("""
    SELECT count(*) FROM protein_go_annotations g
    JOIN uniprot_annotations a USING (source_file, fasta_id)
    WHERE a.status = 'not_found'
""").fetchone()[0]
assert leaked == 0

wrong_accession = conn.execute("""
    SELECT count(*) FROM uniprot_annotations a JOIN proteins p USING (source_file, fasta_id)
    WHERE a.uniprot_accession <> p.uniprot_accession
""").fetchone()[0]
assert wrong_accession == 0
print("anotações só de accessions devolvidos pela fonte, sempre o da própria proteína")


aspect | anotacoes | termos
molecular_function | 2577 | 239
biological_process | 2215 | 539
cellular_component | 2080 | 139

go_id | termo | proteinas
GO:0090729 | toxin activity | 492
GO:0003677 | DNA binding | 222
GO:0003905 | alkylbase DNA N-glycosylase activity | 213
GO:0030170 | pyridoxal phosphate binding | 87
GO:0019888 | protein phosphatase regulator activity | 69
anotações só de accessions devolvidos pela fonte, sempre o da própria proteína


## Resumo

O mesmo resumo que a extração imprime no final, agora com `assert` sobre as contagens.


In [6]:
summary = pipeline.database_summary(conn)
for key, value in summary.items():
    print(f"{key:>22}: {value}")

assert summary["proteins"] == summary["with_features"] == summary["with_composition"]
assert summary["proteins"] == summary["with_dipeptides"]
assert summary["orphans"] == 0
assert (
    summary["matched"] + summary["not_found"] + summary["unqueried"]
    + summary["without_accession"] == summary["proteins"]
)

conn.close()
print("\nvalidação concluída; a base não foi alterada")


              proteins: 2000
     without_accession: 0
         with_features: 2000
      with_composition: 2000
       with_dipeptides: 2000
      feature_versions: 2
               matched: 2000
             not_found: 0
             unqueried: 0
               go_rows: 6872
      proteins_with_go: 1680
          go_by_aspect: {'biological_process': 2215, 'cellular_component': 2080, 'molecular_function': 2577}
                issues: {'noncanonical_residues': 15}
               orphans: 0

validação concluída; a base não foi alterada
